# 04b Aggregate-data placebo / permutation robustness check

> **严格命名与边界**
> - 本分析的正式名称是 **aggregate-data placebo/permutation robustness check（聚合数据安慰剂/置换鲁棒性检查）**，**不是 AA Test，也不能替代 cookie 级 AA Test**：真实处理分配单位是 cookie，而数据只有日粒度聚合，无法重建用户级随机化。
> - 目的：在"无真实处理效应"的零假设下，检验主分析已实现的 **Gross/Net 转化 Z 检验管线**本身的假阳性行为，并用经验零分布为观测统计量提供交叉印证。
> - **exchangeability 处理**：日数据存在星期季节性（周末流量显著低），朴素整表打乱标签会破坏组内星期结构；故采用 **same-weekday block permutation（只在同一星期几的日期块内置换组标签）**，每块保持 Control/Experiment 行数不变。
> - 局限：same-weekday block 只控制了星期季节性；仅 23 个日 cluster、无法处理日序自相关/共同冲击，更无法验证 cookie 级分流（后者前序的 SRM/invariant 已做聚合层面检查）。
> - 本检查**只作鲁棒性佐证，不改写主分析结论，也不做非劣效判定**。

In [1]:
# 加载 23 天 outcome 窗与前序观测统计量
import json, tomllib
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path.cwd()
CFG = tomllib.load(open(ROOT/"config"/"analysis_config.toml","rb"))
EXP = CFG["exploratory"]
B = CFG["exploratory"]["permutation_b"]; SEED = EXP["random_seed"]
assert EXP["permutation_method"].startswith("same-weekday")
long = pd.read_csv(ROOT/"data/processed/daily_long.csv", parse_dates=["Date"])
ow = (long[long["OutcomeComplete"]].copy()
      .sort_values(["Weekday","Group","Date"]).reset_index(drop=True))
main = json.loads((ROOT/"data"/"processed"/"main_effects.json").read_text(encoding="utf-8"))
z_obs = {"GrossConversion": main["z_tests"]["GrossConversion"]["z"],
         "NetConversion": main["z_tests"]["NetConversion"]["z"]}
print("B =", B, "| seed =", SEED, "| rows =", len(ow))
print("observed Z from phase-4 pipeline:", {k: round(v,4) for k,v in z_obs.items()})
# 星期块行数必须两组对称（same-weekday block 置换的前提）
wc = ow.groupby(["Weekday","Group"]).size().unstack()
print(wc)
assert (wc["Control"] == wc["Experiment"]).all()

B = 10000 | seed = 20260831 | rows = 46
observed Z from phase-4 pipeline: {'GrossConversion': -4.7018, 'NetConversion': -1.4192}
Group    Control  Experiment
Weekday                     
Fri            3           3
Mon            3           3
Sat            4           4
Sun            4           4
Thu            3           3
Tue            3           3
Wed            3           3


## 置换设计（预先定义统计量，写入 config）
- **统计量（与主分析管线完全一致）**：双样本比例 pooled-Z = (p_E−p_C)/√(p̄(1−p̄)(1/n_C+1/n_E))。
- **零假设操作化**：同星期块内组标签可交换 → 每次置换在每个 Weekday 块内把组标签随机重排（块内两组行数相等），重算总量与 Z。
- 重复 B=10,000 次得到 Gross/Net 的零分布；经验双侧 p=(1+#|Z_perm|≥|Z_obs|)/(B+1)；并统计零分布在 α=0.05 临界值 ±1.96 之外的比例（经验假阳性率，理论应≈0.05）。

In [2]:
# same-weekday block permutation（向量化）
def z_from_sums(x_c, n_c, x_e, n_e):
    p_c, p_e = x_c/n_c, x_e/n_e
    p = (x_c+x_e)/(n_c+n_e)
    se = np.sqrt(p*(1-p)*(1/n_c + 1/n_e))
    return (p_e-p_c)/se

rng = np.random.default_rng(SEED)
weekdays = ow["Weekday"].unique()
blocks = {w: ow.index[ow["Weekday"] == w].to_numpy() for w in weekdays}

def run_perm(num_col):
    num = ow[num_col].to_numpy(float); den = ow["Clicks"].to_numpy(float)
    grp = (ow["Group"] == "Experiment").to_numpy()
    null_z = np.empty(B); ne_arr = np.empty(B)
    for b in range(B):
        is_e = grp.copy()
        for w, idx in blocks.items():
            # 块内重排标签（保持 E 的行数不变）
            is_e[idx] = rng.permutation(is_e[idx])
        xc, nc = num[~is_e].sum(), den[~is_e].sum()
        xe, ne = num[is_e].sum(), den[is_e].sum()
        null_z[b] = z_from_sums(xc, nc, xe, ne); ne_arr[b] = ne
    return null_z, ne_arr

null_g, ne_g = run_perm("Enrollments")
null_n, ne_n = run_perm("Payments")
print("null Gross: mean=%.4f sd=%.4f" % (null_g.mean(), null_g.std(ddof=1)))
print("null Net  : mean=%.4f sd=%.4f" % (null_n.mean(), null_n.std(ddof=1)))

null Gross: mean=0.0063 sd=3.1413
null Net  : mean=-0.0233 sd=2.5384


In [3]:
# 方法诊断：零分布为何比 N(0,1) 宽？——整天重分配会让组分母(clicks)大幅波动
obs_e_clicks = int(ow.loc[ow.Group=="Experiment","Clicks"].sum())
print("观测 Experiment clicks =", obs_e_clicks)
print("置换后 Experiment clicks: mean=%.0f sd=%.0f min=%d max=%d"
      % (ne_g.mean(), ne_g.std(ddof=1), ne_g.min(), ne_g.max()))
print("组分母波动系数 CV=%.3f（真实 cookie 随机化下两组每天都同时存在、组总量近乎固定，SRM 已验证）"
      % (ne_g.std(ddof=1)/ne_g.mean()))
print("对照：主分析 cluster/iid SE 比值 Gross=3.03、Net=2.58；置换零分布 SD 3.14/2.54 与之同量级，")
print("      说明宽度来自日间异质性+组分母波动，而非管线错误（零分布均值≈0，统计量计算正确）。")

观测 Experiment clicks = 17260
置换后 Experiment clicks: mean=17275 sd=168 min=16704 max=17900
组分母波动系数 CV=0.010（真实 cookie 随机化下两组每天都同时存在、组总量近乎固定，SRM 已验证）
对照：主分析 cluster/iid SE 比值 Gross=3.03、Net=2.58；置换零分布 SD 3.14/2.54 与之同量级，
      说明宽度来自日间异质性+组分母波动，而非管线错误（零分布均值≈0，统计量计算正确）。


In [4]:
# 经验 p、α=0.05 经验拒绝率、与主检验 Z交叉印证
zcrit = 1.959963984540054
results = {}
for name, null_z, obs in [("GrossConversion", null_g, z_obs["GrossConversion"]),
                          ("NetConversion", null_n, z_obs["NetConversion"])]:
    emp_p = (1 + np.sum(np.abs(null_z) >= abs(obs)))/(B+1)
    fpr = np.mean(np.abs(null_z) > zcrit)
    results[name] = {"observed_z": float(obs),
                     "perm_empirical_p_two_sided": float(emp_p),
                     "null_mean": float(null_z.mean()), "null_sd": float(null_z.std(ddof=1)),
                     "empirical_rejection_rate_at_alpha05": float(fpr),
                     "z_pipeline_p": main["z_tests"][name]["p_value"]}
    print(f"=== {name} ===")
    print(f" observed |Z|={abs(obs):.4f}; permutation empirical p={emp_p:.4f}; "
          f"phase-4 Z-test p={main['z_tests'][name]['p_value']:.4g}")
    print(f" null 经验假阳性率@α=0.05 = {fpr:.4f}（设计值 0.05；95%蒙特卡洛包络约 "
          f"{0.05-1.96*np.sqrt(.05*.95/B):.3f}~{0.05+1.96*np.sqrt(.05*.95/B):.3f}）")

=== GrossConversion ===
 observed |Z|=4.7018; permutation empirical p=0.1354; phase-4 Z-test p=2.578e-06
 null 经验假阳性率@α=0.05 = 0.5494（设计值 0.05；95%蒙特卡洛包络约 0.046~0.054）
=== NetConversion ===
 observed |Z|=1.4192; permutation empirical p=0.5874; phase-4 Z-test p=0.1558
 null 经验假阳性率@α=0.05 = 0.4526（设计值 0.05；95%蒙特卡洛包络约 0.046~0.054）


## 同日配对翻转（并列敏感性） swap C/E within each date, fixed daily margins

> **SENSITIVITY ONLY — 主分析仍锁定 same-weekday block，本节不改主结论。**
> - 做法：每个日期都有配对的 Control/Experiment 两行；对每个日期以 0.5 概率整体交换两行组标签 → 两组在**每一天都同时存在、组日边际固定**，更贴近真实 cookie 分流结构（每天同时向两组分流、组总量近乎固定）。
> - 目的：展示**可交换性假设如何改变零分布宽度与经验 p**；它**不是**随机化质量验证、**不是 AA Test**，也不替代主置换。
> - 使用独立随机数流（seed+1），保证 same-weekday block 主结果逐位不变。

In [5]:
# 同日配对翻转（独立 RNG，B=10000）
rng2 = np.random.default_rng(SEED + 1)
dates = ow["Date"].to_numpy()
pos_c = np.array([np.where((dates == d) & (ow["Group"].to_numpy() == "Control"))[0][0]
                  for d in np.unique(dates)])
pos_e = np.array([np.where((dates == d) & (ow["Group"].to_numpy() == "Experiment"))[0][0]
                  for d in np.unique(dates)])
n_dates = len(np.unique(dates))

def paired_perm(num_col):
    num = ow[num_col].to_numpy(float); den = ow["Clicks"].to_numpy(float)
    zs = np.empty(B)
    for b in range(B):
        flip = rng2.random(n_dates) < 0.5
        is_e = np.zeros(len(ow), bool)
        is_e[pos_e] = ~flip; is_e[pos_c] = flip
        xc, nc = num[~is_e].sum(), den[~is_e].sum()
        xe, ne = num[is_e].sum(), den[is_e].sum()
        zs[b] = z_from_sums(xc, nc, xe, ne)
    return zs

pair_g = paired_perm("Enrollments")
pair_n = paired_perm("Payments")
paired = {}
for name, pz, obs in [("GrossConversion", pair_g, z_obs["GrossConversion"]),
                      ("NetConversion", pair_n, z_obs["NetConversion"])]:
    paired[name] = {"empirical_p": float((1 + np.sum(np.abs(pz) >= abs(obs)))/(B+1)),
                    "null_sd": float(pz.std(ddof=1)),
                    "fpr_at_05": float(np.mean(np.abs(pz) > zcrit))}
    print(f"{name}: paired-date null sd={pz.std(ddof=1):.3f}, emp p={paired[name]['empirical_p']:.4f}, "
          f"FPR@.05={paired[name]['fpr_at_05']:.4f}")

GrossConversion: paired-date null sd=1.577, emp p=0.0015, FPR@.05=0.2201
NetConversion: paired-date null sd=1.840, emp p=0.4469, FPR@.05=0.2956


In [6]:
# 三法并排小表：iid-Z/ same-weekday block（主置换）/ paired-date（敏感性）
rows3 = []
for name in ["GrossConversion","NetConversion"]:
    rows3.append({"metric": name, "method": "iid-Z (phase-4 primary test)",
                  "obs_Z": results[name]["observed_z"], "empirical_p": results[name]["z_pipeline_p"],
                  "null_sd": 1.0, "fpr_at_05": 0.05})
    rows3.append({"metric": name, "method": "same-weekday block (primary permutation)",
                  "obs_Z": results[name]["observed_z"], "empirical_p": results[name]["perm_empirical_p_two_sided"],
                  "null_sd": results[name]["null_sd"], "fpr_at_05": results[name]["empirical_rejection_rate_at_alpha05"]})
    rows3.append({"metric": name, "method": "paired-date swap (SENSITIVITY)",
                  "obs_Z": results[name]["observed_z"], "empirical_p": paired[name]["empirical_p"],
                  "null_sd": paired[name]["null_sd"], "fpr_at_05": paired[name]["fpr_at_05"]})
three = pd.DataFrame(rows3)
def _fmt(x):
    return f"{x:.3g}" if 0 < x < 1e-3 else f"{x:.4f}"
print(three.to_string(index=False, float_format=_fmt))

         metric                                   method   obs_Z  empirical_p  null_sd  fpr_at_05
GrossConversion             iid-Z (phase-4 primary test) -4.7018     2.58e-06   1.0000     0.0500
GrossConversion same-weekday block (primary permutation) -4.7018       0.1354   3.1413     0.5494
GrossConversion           paired-date swap (SENSITIVITY) -4.7018       0.0015   1.5771     0.2201
  NetConversion             iid-Z (phase-4 primary test) -1.4192       0.1558   1.0000     0.0500
  NetConversion same-weekday block (primary permutation) -1.4192       0.5874   2.5384     0.4526
  NetConversion           paired-date swap (SENSITIVITY) -1.4192       0.4469   1.8397     0.2956


In [7]:
# 敏感性对照图：两种聚合零分布 vs N(0,1)
grid = np.linspace(-6, 6, 400)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=150, sharey=True)
for ax, name, blk, pr in zip(axes, ["GrossConversion","NetConversion"], [null_g, null_n], [pair_g, pair_n]):
    ax.hist(blk, bins=60, density=True, alpha=.55, color="#4c78a8", label="same-weekday block (primary)")
    ax.hist(pr, bins=60, density=True, histtype="step", color="#e69f00", lw=1.6, label="paired-date (sensitivity)")
    ax.plot(grid, stats.norm.pdf(grid), color="black", lw=1.2, label="N(0,1) iid")
    ax.axvline(z_obs[name], color="#d62728", lw=1.5, label=f"obs Z={z_obs[name]:.2f}")
    ax.set_title(name); ax.set_xlabel("null Z"); ax.legend(fontsize=7); ax.grid(alpha=.3)
axes[0].set_ylabel("density")
fig.suptitle("Exchangeability assumption vs null width (aggregate-data check; not an AA test)")
fig.tight_layout(); fig.savefig(ROOT/"reports"/"figures"/"fig_permutation_sensitivity.png"); plt.close(fig)
print("saved reports/figures/fig_permutation_sensitivity.png")

saved reports/figures/fig_permutation_sensitivity.png


In [8]:
# 零分布图：观测统计量与 ±1.96 临界线
grid = np.linspace(-6, 6, 400)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=150, sharey=True)
for ax, name, nz in zip(axes, ["GrossConversion","NetConversion"], [null_g, null_n]):
    ax.hist(nz, bins=60, color="#4c78a8", alpha=.75, density=True, label="same-weekday-block null")
    ax.plot(grid, stats.norm.pdf(grid), color="black", lw=1.3, label="N(0,1) iid reference")
    ax.axvline(-zcrit, color="gray", ls=":", lw=1.2); ax.axvline(zcrit, color="gray", ls=":", lw=1.2)
    ax.axvline(results[name]["observed_z"], color="#d62728", lw=1.6,
               label=f"observed Z={results[name]['observed_z']:.2f}")
    ax.set_title(f"{name} (null sd={nz.std(ddof=1):.2f})")
    ax.set_xlabel("permutation null Z"); ax.legend(fontsize=7); ax.grid(alpha=.3)
axes[0].set_ylabel("density")
fig.suptitle("Aggregate-data same-weekday-block permutation null (B=10,000; not an AA test)")
fig.tight_layout(); fig.savefig(ROOT/"reports"/"figures"/"fig_permutation_null.png"); plt.close(fig)
print("saved reports/figures/fig_permutation_null.png")

saved reports/figures/fig_permutation_null.png


In [9]:
# 落盘 + 回读验证
out = {"name": "aggregate-data placebo/permutation robustness check",
       "not_aa_test": True,
       "method": "same-weekday block permutation of group labels within weekday (balanced blocks)",
       "exchangeability_note": "controls weekday seasonality; cannot replicate cookie-level randomization; 23 day-clusters only",
       "B": B, "seed": SEED, "window": "23-day outcome window",
       "statistic": "two-sample pooled two-proportion Z (identical to phase-4 pipeline)",
       "null_overdispersion": {"null_sd_gross": float(null_g.std(ddof=1)),
                               "null_sd_net": float(null_n.std(ddof=1)),
                               "cause": "whole-day reassignment fluctuates group click margins and folds day-level overdispersion into null; not a pipeline bug (null mean ~0)",
                               "permuted_experiment_clicks_mean": float(ne_g.mean()),
                               "permuted_experiment_clicks_sd": float(ne_g.std(ddof=1))},
       "sensitivity_paired_date": {"label": "SENSITIVITY ONLY - not primary, not an AA test",
                                   "seed": SEED+1, "B": B, "results": paired},
       "three_method_table": rows3,
       "results": results}
p = ROOT/"data"/"processed"/"permutation_check.json"
p.write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8")
back = json.loads(p.read_text(encoding="utf-8"))
assert back["results"]["GrossConversion"]["observed_z"] == results["GrossConversion"]["observed_z"]
print("permutation_check.json written & re-read OK")

permutation_check.json written & re-read OK


## 小结（只作鲁棒性佐证）
1. **管线机制正确**：same-weekday block 置换零分布关于 0 对称（均值≈0），统计量计算无误。
2. **但零分布比 N(0,1) 宽（SD≈3.14/2.54，α=0.05 名义临界线下经验拒绝率 0.55/0.45）**：原因不是星期季节性（朴素整表置换同样宽），而是整天重分配使两组 clicks 分母大幅波动、并把日间过度离散折进零分布；真实 cookie 随机化下两组每天同时存在、组总量近乎固定（SRM 已验证）。这与相应阶段"cluster SE 为 iid 的 2.6–3.0 倍"完全同量级、互相印证。
3. **交叉印证结论（按同一口径）**：
   - Net：iid-Z p=0.156、置换经验 p≈0.59，**两条路径都不显著**，结论稳健；
   - Gross：iid-Z p=2.58e-6 极强，但在"以天为可交换单位"的置换零分布下经验 p≈0.135——**再次说明 Gross 的显著性强度依赖 click 相互独立假设**，不得叙述为无条件显著（与 D4-2/D4-3、README Limitations 一致）。
4. **边界**：这是**聚合数据** placebo/permutation robustness check，**不是 AA Test、不能替代 cookie 级 AA**；same-weekday block 只缓解星期季节性，23 个日 cluster、日序相关无法处理。不改写主结论、不做非劣效判定。
5. **并列敏感性（已获批准）**：同日配对翻转保持组日边际固定，零分布 SD 介于 iid 与 block 之间（见三法对照表）；它直观展示"可交换性单位假设越接近真实 cookie 分流，零分布越窄、经验 p 越接近 iid-Z"。该节仅为 sensitivity，不是随机化质量验证、不是 AA Test、不改主结论。